# 01 — Kotekar Data Preparation (LOCAL)
## WavSent-MTL · Tasks 1.9–1.13

**Pipeline steps covered (Steps 1–3 for Kotekar):**
1. Load Nifty50 OHLCV from `data/raw/nifty50_ohlcv.csv`
2. Load pre-computed FinBERT sentiment from `data/finbert_outputs/kotekar_sentiment.csv`
3. Merge on trading date, fill missing sentiment with 0.0
4. Verify zero NaNs in polarity_mean
5. Save → `data/processed/kotekar/merged_data.csv`

**Hard rules:**
- No shuffling at any point
- Never compute features here — that is notebook 03
- Do NOT re-run FinBERT

In [1]:
import sys
import os

# Add project root to path (run from notebooks/ or from root)
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from config.config import CONFIG

print('CONFIG loaded.')
print('Kotekar period:', CONFIG['kotekar_start'], '->', CONFIG['kotekar_end'])

CONFIG loaded.
Kotekar period: 2020-01-01 -> 2024-05-31


## Step 1 — Load Nifty50 OHLCV

In [2]:
from src.data.loader import load_price_data

price_df = load_price_data()

print(f'Price shape:  {price_df.shape}')
print(f'Date range:   {price_df["Date"].min().date()} -> {price_df["Date"].max().date()}')
print(f'Columns:      {price_df.columns.tolist()}')
print(f'NaN total:    {price_df.isnull().sum().sum()}')
price_df.head(3)

Price shape:  (1826, 6)
Date range:   2017-01-02 -> 2024-05-31
Columns:      ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
NaN total:    0


,Date,Open,High,Low,Close,Volume
0,2017-01-02,8210.099609,8212.000000,8133.799805,8179.50,118300.0
1,2017-01-03,8196.049805,8219.099609,8148.600098,8192.25,127300.0
2,2017-01-04,8202.650391,8218.500000,8180.899902,8190.50,132400.0


## Step 2 — Load Kotekar Sentiment

In [3]:
from src.data.loader import load_kotekar_sentiment

kotekar_sent = load_kotekar_sentiment()

print(f'Shape:      {kotekar_sent.shape}')
print(f'Columns:    {kotekar_sent.columns.tolist()}')
print(f'Date range: {kotekar_sent["date"].min()} -> {kotekar_sent["date"].max()}')
print(f'NaN count:  {kotekar_sent.isnull().sum().sum()}')
kotekar_sent.head(3)

Shape:      (1092, 2)
Columns:    ['date', 'polarity_mean']
Date range: 2020-01-01 00:00:00 -> 2024-05-31 00:00:00
NaN count:  0


,date,polarity_mean
0,2020-01-01,0.0
1,2020-01-02,0.0
2,2020-01-03,0.0


## Step 3 — Merge Price + Kotekar Sentiment

In [4]:
from src.data.loader import merge_kotekar

df_kotekar = merge_kotekar(price_df, kotekar_sent)

print(f'Merged shape: {df_kotekar.shape}')
print(f'Columns:      {df_kotekar.columns.tolist()}')
print(f'Date range:   {df_kotekar["Date"].min().date()} -> {df_kotekar["Date"].max().date()}')
df_kotekar.head(3)

Merged shape: (1092, 7)
Columns:      ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'polarity_mean']
Date range:   2020-01-01 -> 2024-05-31


,Date,Open,High,Low,Close,Volume,polarity_mean
0,2020-01-01,12202.15039,12222.20020,12165.29980,12182.50000,304100.0,0.0
1,2020-01-02,12198.54980,12289.90039,12195.25000,12282.20020,407700.0,0.0
2,2020-01-03,12261.09961,12265.59961,12191.34961,12226.65039,428800.0,0.0


## Task 1.11 — Verify Missing Dates Filled with polarity_mean = 0

In [5]:
# Check 1: No NaN in polarity_mean
nan_count = df_kotekar['polarity_mean'].isnull().sum()
assert nan_count == 0, f'Found {nan_count} NaN in polarity_mean!'
print(f'NaN in polarity_mean: {nan_count}  OK')

# Check 2: Zero-fill statistics
zero_filled = (df_kotekar['polarity_mean'] == 0.0).sum()
nonzero = (df_kotekar['polarity_mean'] != 0.0).sum()
print(f'Zero-filled rows:     {zero_filled}')
print(f'Non-zero sentiment:   {nonzero}')

# Check 3: No NaN in OHLCV
ohlcv_nan = df_kotekar[['Open', 'High', 'Low', 'Close', 'Volume']].isnull().sum().sum()
assert ohlcv_nan == 0, f'OHLCV has {ohlcv_nan} NaN values'
print(f'OHLCV NaN total:      {ohlcv_nan}  OK')

# Check 4: Period boundaries
assert str(df_kotekar['Date'].min().date()) >= CONFIG['kotekar_start']
assert str(df_kotekar['Date'].max().date()) <= CONFIG['kotekar_end']
print(f'Period verified:      {CONFIG["kotekar_start"]} -> {CONFIG["kotekar_end"]}  OK')

NaN in polarity_mean: 0  OK
Zero-filled rows:     464
Non-zero sentiment:   628
OHLCV NaN total:      0  OK
Period verified:      2020-01-01 -> 2024-05-31  OK


## Task 1.12 — Save merged_data.csv

In [6]:
out_dir = os.path.join(project_root, CONFIG['kotekar_processed_dir'])
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, 'merged_data.csv')
df_kotekar.to_csv(out_path, index=False)

# Reload and verify
df_check = pd.read_csv(out_path)
assert df_check.shape == df_kotekar.shape, 'Save/reload shape mismatch'

print(f'Saved:  {out_path}')
print(f'Shape:  {df_kotekar.shape}')
print('Reload: OK')
df_check.tail(3)

Saved:  d:/WavSent-MTL/data/processed/kotekar/merged_data.csv
Shape:  (1092, 7)
Reload: OK


,Date,Open,High,Low,Close,Volume,polarity_mean
1089,2024-05-29,22762.75000,22825.50,22685.44922,22704.69922,269900.0,0.0
1090,2024-05-30,22617.44922,22705.75,22417.00000,22488.65039,373400.0,0.0
1091,2024-05-31,22568.09961,22653.75,22465.09961,22530.69922,572100.0,0.0


## Summary

In [7]:
print('=' * 50)
print('Notebook 01 -- Kotekar Data Prep: COMPLETE')
print('=' * 50)
print(f'  Rows:           {len(df_kotekar)}')
print(f'  Date range:     {df_kotekar["Date"].min().date()} -> {df_kotekar["Date"].max().date()}')
print(f'  polarity_mean:  min={df_kotekar["polarity_mean"].min():.4f}  max={df_kotekar["polarity_mean"].max():.4f}')
print(f'  Zero-filled:    {(df_kotekar["polarity_mean"] == 0.0).sum()} rows')
print(f'  Output:         {out_path}')
print()
print('Next: run 02_data_prep_kaggle.ipynb')

Notebook 01 -- Kotekar Data Prep: COMPLETE
  Rows:           1092
  Date range:     2020-01-01 -> 2024-05-31
  polarity_mean:  min=-0.9684  max=0.9438
  Zero-filled:    464 rows
  Output:         d:/WavSent-MTL/data/processed/kotekar/merged_data.csv

Next: run 02_data_prep_kaggle.ipynb
